# CoKeeper Naive Bayes Pipeline

**Goal:** Implement Naive Bayes classifier to approach CatBoost (88.8%) and XGBoost (91.1%) accuracy

**Strategy:**
- Test multiple NB variants: MultinomialNB, ComplementNB, GaussianNB
- Use same feature engineering (TF-IDF 500 features + Vendor Intelligence)
- Apply dual model architecture (matched/unmatched vendors)
- Optimize with feature selection, alpha tuning, and ensemble methods

**Expected Accuracy:** 75-85% baseline, targeting 85%+ with optimization

In [1]:
# Setup and imports
import pandas as pd
import numpy as np
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB, ComplementNB, GaussianNB
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import VotingClassifier
from src.features.vendor_intelligence import VendorIntelligence
from src.features.rule_based_classifier import RuleBasedClassifier
from src.features.post_prediction_validator import PostPredictionValidator
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports complete")

✓ Imports complete


## Step 1: Load and Classify General Ledger Data

Using the same data pipeline as XGBoost for fair comparison

In [2]:
# Load General Ledger
gl = pd.read_csv('/Users/gagepiercegaubert/Desktop/career_projects/co-keeper/CSV_data/by_year/General_ledger_2023.csv')

print(f"Loaded General Ledger: {len(gl)} rows")
print(f"Columns: {list(gl.columns)}")

Loaded General Ledger: 3218 rows
Columns: ['Date', 'Transaction Type', 'Num', 'Name', 'Memo/Description', 'Account', 'Debit', 'Credit', 'Balance']


In [3]:
# Classify accounts by type (same function as other pipelines)
def classify_account(account_str):
    """
    Classify QuickBooks account based on account code.

    QuickBooks standard numbering:
    10000-19999: Assets
    20000-29999: Liabilities
    30000-39999: Equity
    40000-49999: Income
    50000-59999: Cost of Goods Sold
    60000-79999: Operating Expenses
    80000-89999: Other Income
    90000-99999: Other Expenses
    """
    if pd.isna(account_str):
        return 'UNKNOWN', '', ''

    account_str = str(account_str).strip()
    parts = account_str.split(' ', 1)
    code_str = parts[0]
    name = parts[1] if len(parts) > 1 else account_str

    try:
        code_num = float(code_str.split('.')[0])
    except ValueError:
        return 'UNKNOWN', code_str, name

    if code_num < 20000:
        acct_type = 'ASSET'
    elif code_num < 30000:
        acct_type = 'LIABILITY'
    elif code_num < 40000:
        acct_type = 'EQUITY'
    elif code_num < 50000:
        acct_type = 'INCOME'
    elif code_num < 60000:
        acct_type = 'COGS'
    elif code_num < 80000:
        acct_type = 'EXPENSE'
    elif code_num < 90000:
        acct_type = 'OTHER_INCOME'
    elif code_num < 100000:
        acct_type = 'OTHER_EXPENSE'
    else:
        acct_type = 'UNKNOWN'

    return acct_type, code_str, name

# Apply classification
gl[['account_type', 'account_code', 'account_name']] = gl['Account'].apply(
    lambda x: pd.Series(classify_account(x))
)

print("\nAccount type distribution:")
print(gl['account_type'].value_counts())


Account type distribution:
account_type
LIABILITY    1128
EXPENSE       590
COGS          547
UNKNOWN       421
ASSET         395
INCOME        124
EQUITY         13
Name: count, dtype: int64


## Step 2: Extract Clean Training Dataset

In [4]:
# Filter to ONLY expense/income accounts
CATEGORY_TYPES = ['EXPENSE', 'COGS', 'INCOME', 'OTHER_INCOME', 'OTHER_EXPENSE']

clean = gl[gl['account_type'].isin(CATEGORY_TYPES)].copy()

print(f"Filtered to category accounts: {len(clean)} rows")

# Get memo column
memo_cols = [c for c in gl.columns if 'memo' in c.lower() or 'description' in c.lower()]
memo_col = memo_cols[0] if memo_cols else None

# Build description from vendor name + memo
clean['vendor_name'] = clean['Name'].fillna('').str.strip()
clean['memo'] = clean[memo_col].fillna('').str.strip() if memo_col else ''
clean['description'] = clean.apply(
    lambda row: ' '.join(filter(None, [row['vendor_name'], row['memo']])).strip(),
    axis=1
)

# Amount: take whichever of Debit/Credit is non-zero
clean['amount'] = clean['Debit'].fillna(0) + clean['Credit'].fillna(0)

# Category = the Transaction Type column (what we want to predict)
clean['category_true'] = clean['Transaction Type']

# Parse date
clean['date'] = pd.to_datetime(clean['Date'], errors='coerce')

# Select final columns
training_data = clean[[
    'date', 'description', 'vendor_name', 'amount',
    'category_true', 'account_code'
]].copy()

# Remove zero-amount and empty descriptions
training_data = training_data[training_data['amount'] > 0]
training_data = training_data[training_data['description'].str.strip().str.len() > 0]

print(f"Clean training dataset: {len(training_data)} transactions")
print(f"Unique categories: {training_data['category_true'].nunique()}")

# Remove categories with <10 examples (need at least 10 for 70/15/15 stratified split)
MIN_EXAMPLES = 10
cat_counts = training_data['category_true'].value_counts()
small_cats = cat_counts[cat_counts < MIN_EXAMPLES].index

if len(small_cats) > 0:
    print(f"\nRemoving {len(small_cats)} categories with <{MIN_EXAMPLES} examples")
    training_data = training_data[~training_data['category_true'].isin(small_cats)]

print(f"Final dataset: {len(training_data)} transactions, {training_data['category_true'].nunique()} categories")

Filtered to category accounts: 1261 rows
Clean training dataset: 1246 transactions
Unique categories: 7

Removing 1 categories with <10 examples
Final dataset: 1245 transactions, 6 categories


## Step 3: Create Train/Val/Test Splits (Same as XGBoost)

In [5]:
# Stratified split (70% train, 15% val, 15% test) - Same random seed as XGBoost
total = len(training_data)

# First split: train (70%) vs temp (30%)
train, temp = train_test_split(
    training_data,
    test_size=0.30,
    stratify=training_data['category_true'],
    random_state=42  # Same seed as other pipelines
)

# Second split: val (50% of temp = 15% overall) vs test (50% of temp = 15% overall)
val, test = train_test_split(
    temp,
    test_size=0.50,
    stratify=temp['category_true'],
    random_state=42
)

train = train.reset_index(drop=True)
val = val.reset_index(drop=True)
test = test.reset_index(drop=True)

print(f"Train: {len(train)} ({len(train)/total*100:.1f}%)")
print(f"Val:   {len(val)} ({len(val)/total*100:.1f}%)")
print(f"Test:  {len(test)} ({len(test)/total*100:.1f}%)")

Train: 871 (70.0%)
Val:   187 (15.0%)
Test:  187 (15.0%)


## Step 4: Build TF-IDF Features (500 features like XGBoost)

In [6]:
# TF-IDF vectorization on descriptions
tfidf = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2),
    min_df=2,
    stop_words='english'
)

train_tfidf = tfidf.fit_transform(train['description'].fillna(''))
val_tfidf = tfidf.transform(val['description'].fillna(''))
test_tfidf = tfidf.transform(test['description'].fillna(''))

# Convert to DataFrame (NB works with arrays, but keeping consistent)
tfidf_cols = [f'tfidf_{i}' for i in range(train_tfidf.shape[1])]
train_features = pd.DataFrame(train_tfidf.toarray(), columns=tfidf_cols)
val_features = pd.DataFrame(val_tfidf.toarray(), columns=tfidf_cols)
test_features = pd.DataFrame(test_tfidf.toarray(), columns=tfidf_cols)

# Add amount (normalized for NB)
# Naive Bayes works better with non-negative features, so we'll use log transform
train_features['amount_log'] = np.log1p(train['amount'].values)
val_features['amount_log'] = np.log1p(val['amount'].values)
test_features['amount_log'] = np.log1p(test['amount'].values)

print(f"TF-IDF features: {len(tfidf_cols)}")
print(f"Total features: {train_features.shape[1]} (TF-IDF + amount_log)")

TF-IDF features: 500
Total features: 501 (TF-IDF + amount_log)


## Step 5: Integrate Vendor Intelligence System

In [ ]:
# Initialize and train Vendor Intelligence
vi = VendorIntelligence(
    exact_min_consistency=0.80,
    exact_min_occurrences=2,
    use_merchant_normalization=True
)

train_vi_data = train[['vendor_name', 'description', 'amount', 'category_true']].copy()
vi.fit(train_vi_data)

MerchantNormalizer fitted: 110 aliases learned
VendorIntelligence fitted:
  Level 0 (normalize): 110 merchant aliases learned
  Level 1 (exact): 69 vendor→category mappings
  Level 2 (fuzzy): Ready (69 vendors indexed)
  Level 3 (type):  0 category hints mapped
Vendor Intelligence trained:
  Vendor mappings: 69
  Merchant aliases: 110


In [ ]:
# Apply VI predictions to all splits
def apply_vi(df, vi_model):
    """Apply Vendor Intelligence and return VI features"""
    vi_results = []

    for _, row in df.iterrows():
        result = vi_model.classify(
            vendor_name=row['vendor_name'],
            description=row['description'],
            amount=row['amount']
        )
        vi_results.append({
            'vi_confidence': result.confidence,
            'has_match': 1 if result.match_level != 'none' else 0
        })

    vi_df = pd.DataFrame(vi_results)
    return vi_df

train_vi = apply_vi(train, vi)
val_vi = apply_vi(val, vi)
test_vi = apply_vi(test, vi)

print(f"VI coverage - Train: {train_vi['has_match'].mean()*100:.1f}%, Test: {test_vi['has_match'].mean()*100:.1f}%")

Applying VI to train/val/test...
✓ VI features generated
  Train VI coverage: 74.2%
  Test VI coverage: 66.3%


In [ ]:
# Add binary transportation category features
from src.features.transportation_keywords import detect_transportation_type

def add_transportation_features(df):
    """Add binary indicators for transportation categories"""
    transport_features = pd.DataFrame(index=df.index)

    # Define transportation categories
    categories = ['airline', 'rideshare', 'gas_station', 'parking',
                  'public_transit', 'toll', 'car_rental', 'auto_service']

    # Initialize all to 0
    for cat in categories:
        transport_features[f'is_{cat}'] = 0

    # Detect transportation type for each row
    for idx, row in df.iterrows():
        transport_type = detect_transportation_type(
            description=row['description'],
            vendor_name=row['vendor_name']
        )
        if transport_type:
            transport_features.loc[idx, f'is_{transport_type}'] = 1

    return transport_features

train_transport = add_transportation_features(train)
val_transport = add_transportation_features(val)
test_transport = add_transportation_features(test)

transport_detected = train_transport.sum(axis=1) > 0
print(f"Transportation coverage: {transport_detected.mean()*100:.1f}% ({train_transport.shape[1]} features)")

Extracting transportation features...
✓ Transportation features added
  Train transportation coverage: 3.4%
  Features: 8 binary indicators

  Transportation type distribution in training:
    is_airline: 1 (0.1%)
    is_rideshare: 12 (1.4%)
    is_gas_station: 2 (0.2%)
    is_parking: 4 (0.5%)
    is_public_transit: 11 (1.3%)


In [ ]:
# Initialize rule-based classifier
rule_classifier = RuleBasedClassifier()

def apply_rule_based_classification(df, vi_results=None):
    """Apply rule-based classification with optional VI vendor_type"""
    rb_results = []

    for idx, row in df.iterrows():
        # Get vendor_type from VI if available
        vendor_type = None
        if vi_results is not None and idx < len(vi_results):
            # VI results don't have vendor_type exposed yet, so we'll skip for now
            # Could enhance VI to expose this
            pass

        result = rule_classifier.classify(
            description=row['description'],
            vendor_name=row['vendor_name'],
            amount=row['amount'],
            vendor_type=vendor_type
        )

        rb_results.append({
            'rule_prediction': result.transaction_type if result.transaction_type else '',
            'rule_confidence': result.confidence,
            'has_rule_match': 1 if result.transaction_type else 0,
            'rule_type': result.rule.split('_')[0] if result.transaction_type else 'none'
        })

    return pd.DataFrame(rb_results)

train_rules = apply_rule_based_classification(train)
val_rules = apply_rule_based_classification(val)
test_rules = apply_rule_based_classification(test)

rule_coverage = train_rules['has_rule_match'].mean()
rule_avg_conf = train_rules[train_rules['has_rule_match']==1]['rule_confidence'].mean()
print(f"Rule-based coverage: {rule_coverage*100:.1f}%, Avg confidence: {rule_avg_conf:.1%}")

Applying rule-based classification...
✓ Rule-based classification complete
  Train coverage: 0.8% caught by rules
  Avg confidence (when matched): 95.0%

  Rule type breakdown:
    keyword: 7 (0.8%)

  Predicted transaction types (rule-based):
    Expense: 5
    Deposit: 1
    Bill: 1


## Step 5c: Apply Rule-Based Classification

Pre-process transactions with high-confidence rules before ML model

In [11]:
# Combine TF-IDF + VI + Transportation + Rule-based features
# For rule-based, only use numeric confidence features (not the string predictions)
train_rules_numeric = train_rules[['rule_confidence', 'has_rule_match']]
val_rules_numeric = val_rules[['rule_confidence', 'has_rule_match']]
test_rules_numeric = test_rules[['rule_confidence', 'has_rule_match']]

train_combined = pd.concat([train_features, train_vi, train_transport, train_rules_numeric], axis=1)
val_combined = pd.concat([val_features, val_vi, val_transport, val_rules_numeric], axis=1)
test_combined = pd.concat([test_features, test_vi, test_transport, test_rules_numeric], axis=1)

# Prepare labels
train_labels = train['category_true'].astype(str).to_numpy()
val_labels = val['category_true'].astype(str).to_numpy()
test_labels = test['category_true'].astype(str).to_numpy()

print(f"Combined feature shape: {train_combined.shape}")
print(f"Features breakdown:")
print(f"  - TF-IDF + amount: {train_features.shape[1]}")
print(f"  - Vendor Intelligence: {train_vi.shape[1]}")
print(f"  - Transportation: {train_transport.shape[1]}")
print(f"  - Rule-based: {train_rules_numeric.shape[1]}")
print(f"  - Total: {train_combined.shape[1]}")

Combined feature shape: (871, 513)
Features breakdown:
  - TF-IDF + amount: 501
  - Vendor Intelligence: 2
  - Transportation: 8
  - Rule-based: 2
  - Total: 513


## Step 6: Naive Bayes Model Training

### Experiment 1: Baseline MultinomialNB

In [12]:
# Baseline MultinomialNB - Best for TF-IDF text features
print("Training MultinomialNB (baseline)...")

nb_multi = MultinomialNB(alpha=1.0)  # Default smoothing
nb_multi.fit(train_combined, train_labels)

# Predictions
train_pred_multi = nb_multi.predict(train_combined)
val_pred_multi = nb_multi.predict(val_combined)
test_pred_multi = nb_multi.predict(test_combined)

# Accuracies
train_acc_multi = accuracy_score(train_labels, train_pred_multi)
val_acc_multi = accuracy_score(val_labels, val_pred_multi)
test_acc_multi = accuracy_score(test_labels, test_pred_multi)

print(f"\n✓ MultinomialNB Results:")
print(f"  Train Accuracy: {train_acc_multi:.1%}")
print(f"  Val Accuracy:   {val_acc_multi:.1%}")
print(f"  Test Accuracy:  {test_acc_multi:.1%}")

Training MultinomialNB (baseline)...

✓ MultinomialNB Results:
  Train Accuracy: 94.6%
  Val Accuracy:   93.6%
  Test Accuracy:  93.0%


### Experiment 2: ComplementNB (Better for Imbalanced Classes)

In [13]:
# ComplementNB - Often better for imbalanced datasets
print("Training ComplementNB...")

nb_comp = ComplementNB(alpha=1.0)
nb_comp.fit(train_combined, train_labels)

# Predictions
train_pred_comp = nb_comp.predict(train_combined)
val_pred_comp = nb_comp.predict(val_combined)
test_pred_comp = nb_comp.predict(test_combined)

# Accuracies
train_acc_comp = accuracy_score(train_labels, train_pred_comp)
val_acc_comp = accuracy_score(val_labels, val_pred_comp)
test_acc_comp = accuracy_score(test_labels, test_pred_comp)

print(f"\n✓ ComplementNB Results:")
print(f"  Train Accuracy: {train_acc_comp:.1%}")
print(f"  Val Accuracy:   {val_acc_comp:.1%}")
print(f"  Test Accuracy:  {test_acc_comp:.1%}")

Training ComplementNB...

✓ ComplementNB Results:
  Train Accuracy: 93.2%
  Val Accuracy:   88.2%
  Test Accuracy:  86.6%


### Experiment 3: Hyperparameter Tuning (Alpha)

Test different alpha values for smoothing

In [14]:
# Test different alpha values
print("Testing alpha values for MultinomialNB...")

alphas = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
alpha_results = []

for alpha in alphas:
    nb = MultinomialNB(alpha=alpha)
    nb.fit(train_combined, train_labels)

    val_pred = nb.predict(val_combined)
    test_pred = nb.predict(test_combined)

    val_acc = accuracy_score(val_labels, val_pred)
    test_acc = accuracy_score(test_labels, test_pred)

    alpha_results.append({
        'alpha': alpha,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc
    })

    print(f"  Alpha={alpha:5.2f}: Val={val_acc:.3f}, Test={test_acc:.3f}")

# Find best alpha
best_result = max(alpha_results, key=lambda x: x['val_accuracy'])
print(f"\n✓ Best alpha: {best_result['alpha']:.2f} (Val: {best_result['val_accuracy']:.1%})")

Testing alpha values for MultinomialNB...
  Alpha= 0.01: Val=0.920, Test=0.941
  Alpha= 0.10: Val=0.914, Test=0.936
  Alpha= 0.50: Val=0.941, Test=0.941
  Alpha= 1.00: Val=0.936, Test=0.930
  Alpha= 2.00: Val=0.840, Test=0.818
  Alpha= 5.00: Val=0.706, Test=0.711

✓ Best alpha: 0.50 (Val: 94.1%)


### Experiment 4: Feature Selection (SelectKBest)

Reduce noise by selecting top K features

In [15]:
# Test feature selection with different K values
print("Testing feature selection...")

k_values = [100, 200, 300, 400, 500]  # 500 = all features
k_results = []

for k in k_values:
    if k < train_combined.shape[1]:
        # Select top K features
        selector = SelectKBest(chi2, k=k)
        train_selected = selector.fit_transform(train_combined, train_labels)
        val_selected = selector.transform(val_combined)
        test_selected = selector.transform(test_combined)
    else:
        # Use all features
        train_selected = train_combined
        val_selected = val_combined
        test_selected = test_combined

    # Train with best alpha from previous experiment
    nb = MultinomialNB(alpha=best_result['alpha'])
    nb.fit(train_selected, train_labels)

    val_pred = nb.predict(val_selected)
    test_pred = nb.predict(test_selected)

    val_acc = accuracy_score(val_labels, val_pred)
    test_acc = accuracy_score(test_labels, test_pred)

    k_results.append({
        'k_features': k,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc
    })

    print(f"  K={k:3d} features: Val={val_acc:.3f}, Test={test_acc:.3f}")

# Find best K
best_k_result = max(k_results, key=lambda x: x['val_accuracy'])
print(f"\n✓ Best K: {best_k_result['k_features']} features (Val: {best_k_result['val_accuracy']:.1%})")

Testing feature selection...
  K=100 features: Val=0.941, Test=0.930
  K=200 features: Val=0.936, Test=0.947
  K=300 features: Val=0.936, Test=0.952
  K=400 features: Val=0.941, Test=0.947
  K=500 features: Val=0.941, Test=0.941

✓ Best K: 100 features (Val: 94.1%)


### Experiment 5: Final Optimized Model

Train final model with best hyperparameters

In [16]:
# Train final optimized model
print("Training final optimized MultinomialNB...")

# Use best alpha and K from experiments
best_alpha = best_result['alpha']
best_k = best_k_result['k_features']

if best_k < train_combined.shape[1]:
    selector = SelectKBest(chi2, k=best_k)
    train_final = selector.fit_transform(train_combined, train_labels)
    val_final = selector.transform(val_combined)
    test_final = selector.transform(test_combined)
else:
    train_final = train_combined
    val_final = val_combined
    test_final = test_combined

nb_final = MultinomialNB(alpha=best_alpha)
nb_final.fit(train_final, train_labels)

# Final predictions
train_pred_final = nb_final.predict(train_final)
val_pred_final = nb_final.predict(val_final)
test_pred_final = nb_final.predict(test_final)

# Final accuracies
train_acc_final = accuracy_score(train_labels, train_pred_final)
val_acc_final = accuracy_score(val_labels, val_pred_final)
test_acc_final = accuracy_score(test_labels, test_pred_final)

print(f"\n" + "="*70)
print("NAIVE BAYES FINAL RESULTS")
print("="*70)
print(f"Optimizations: alpha={best_alpha:.2f}, K={best_k} features")
print(f"\n  Train Accuracy: {train_acc_final:.1%}")
print(f"  Val Accuracy:   {val_acc_final:.1%}")
print(f"  Test Accuracy:  {test_acc_final:.1%}")
print("="*70)

Training final optimized MultinomialNB...

NAIVE BAYES FINAL RESULTS
Optimizations: alpha=0.50, K=100 features

  Train Accuracy: 93.5%
  Val Accuracy:   94.1%
  Test Accuracy:  93.0%


In [17]:
# Check how transportation features impact predictions
print("Transportation Feature Analysis:")
print("="*70)

# Check feature importance by looking at selected features
if best_k < train_combined.shape[1]:
    # Get selected feature names
    all_feature_names = train_combined.columns.tolist()
    selected_mask = selector.get_support()
    selected_features = [all_feature_names[i] for i, selected in enumerate(selected_mask) if selected]

    # Count transportation features selected
    transport_selected = [f for f in selected_features if f.startswith('is_')]
    print(f"\nTransportation features selected: {len(transport_selected)}/{train_transport.shape[1]}")
    if transport_selected:
        print(f"  Selected: {', '.join(transport_selected)}")
else:
    print(f"\nAll {train_transport.shape[1]} transportation features used (no feature selection)")

# Analyze transportation coverage in test set
test_with_transport = test_transport.sum(axis=1) > 0
print(f"\nTest set transportation detection:")
print(f"  Transactions with transport features: {test_with_transport.sum()} ({test_with_transport.mean()*100:.1f}%)")

# Check accuracy on transportation vs non-transportation transactions
test_transport_acc = accuracy_score(test_labels[test_with_transport], test_pred_final[test_with_transport])
test_non_transport_acc = accuracy_score(test_labels[~test_with_transport], test_pred_final[~test_with_transport])

print(f"\nAccuracy breakdown:")
print(f"  Transportation transactions:     {test_transport_acc:.1%}")
print(f"  Non-transportation transactions: {test_non_transport_acc:.1%}")
print(f"  Overall test accuracy:           {test_acc_final:.1%}")

# Show which transportation types are most common in errors
if test_with_transport.sum() > 0:
    transport_correct = test_pred_final[test_with_transport] == test_labels[test_with_transport]
    transport_errors = test.iloc[test_with_transport.to_numpy()][~transport_correct]

    if len(transport_errors) > 0:
        print(f"\nTop transportation error patterns (first 5):")
        for idx, row in transport_errors.head(5).iterrows():
            row_idx = row.name  # Get the original index
            test_idx = test.index.get_loc(row_idx) if row_idx in test.index else -1
            if test_idx >= 0:
                pred = test_pred_final[test_idx]
                actual = test_labels[test_idx]
                print(f"  '{row['description'][:50]}'")
                print(f"    Predicted: {pred}, Actual: {actual}")

print("="*70)

Transportation Feature Analysis:

Transportation features selected: 1/8
  Selected: is_rideshare

Test set transportation detection:
  Transactions with transport features: 11 (5.9%)

Accuracy breakdown:
  Transportation transactions:     100.0%
  Non-transportation transactions: 92.6%
  Overall test accuracy:           93.0%


In [18]:
# Analyze rule-based classification impact on test set
print("Rule-Based Classification Analysis:")
print("="*70)

# Check which test transactions would be caught by rules
test_rule_matches = test_rules['has_rule_match'] == 1
test_rule_high_conf = test_rules['rule_confidence'] >= 0.85
test_rule_usable = test_rule_matches & test_rule_high_conf

print(f"\nTest set rule coverage:")
print(f"  Total test transactions: {len(test)}")
print(f"  Matched by rules: {test_rule_matches.sum()} ({test_rule_matches.mean()*100:.1f}%)")
print(f"  High confidence (≥85%): {test_rule_usable.sum()} ({test_rule_usable.mean()*100:.1f}%)")

# Check accuracy of rule-based predictions
if test_rule_usable.sum() > 0:
    rule_predictions_test = test_rules[test_rule_usable]['rule_prediction'].values
    rule_actuals_test = test_labels[test_rule_usable]
    rule_accuracy = accuracy_score(rule_actuals_test, rule_predictions_test)

    print(f"\n  Rule-based accuracy: {rule_accuracy:.1%}")
    print(f"  ML model accuracy: {test_acc_final:.1%}")

    # Calculate hybrid accuracy (rules for high-conf, ML for rest)
    hybrid_predictions = test_pred_final.copy()
    hybrid_predictions[test_rule_usable] = rule_predictions_test
    hybrid_accuracy = accuracy_score(test_labels, hybrid_predictions)

    print(f"  Hybrid accuracy (rules + ML): {hybrid_accuracy:.1%}")

    # Workload reduction
    ml_workload_pct = (1 - test_rule_usable.mean()) * 100
    print(f"\n  ML model workload reduced to: {ml_workload_pct:.1f}%")
    print(f"  (Rules handled {test_rule_usable.mean()*100:.1f}% of transactions)")

# Show which rule types are most effective
if test_rule_matches.sum() > 0:
    print(f"\n  Rule type effectiveness:")
    for rule_type in ['keyword', 'vendor', 'empty']:
        mask = test_rules['rule_type'] == rule_type
        if mask.sum() > 0:
            type_preds = test_rules[mask]['rule_prediction'].values
            type_actuals = test_labels[mask]
            type_acc = accuracy_score(type_actuals, type_preds)
            print(f"    {rule_type}: {mask.sum()} transactions, {type_acc:.1%} accuracy")

# Show rule predictions by transaction type
if test_rule_usable.sum() > 0:
    print(f"\n  Rule predictions by type:")
    rule_pred_dist = test_rules[test_rule_usable]['rule_prediction'].value_counts()
    for trans_type, count in rule_pred_dist.items():
        print(f"    {trans_type}: {count}")

print("="*70)

Rule-Based Classification Analysis:

Test set rule coverage:
  Total test transactions: 187
  Matched by rules: 1 (0.5%)
  High confidence (≥85%): 1 (0.5%)

  Rule-based accuracy: 100.0%
  ML model accuracy: 93.0%
  Hybrid accuracy (rules + ML): 93.6%

  ML model workload reduced to: 99.5%
  (Rules handled 0.5% of transactions)

  Rule type effectiveness:
    keyword: 1 transactions, 100.0% accuracy

  Rule predictions by type:
    Journal Entry: 1


### Rule-Based Classification Impact

Analyze how pre-processing rules reduce ML model load

### Transportation Features Impact

Compare model performance with transportation features added

## Step 7: Confidence Analysis & Tier Distribution

In [19]:
# Get confidence scores (probability of predicted class)
test_proba_final = nb_final.predict_proba(test_final)
test_confidence = test_proba_final.max(axis=1)

# Create results DataFrame
results = test.copy()
results['predicted'] = test_pred_final
results['confidence'] = test_confidence
results['correct'] = results['predicted'] == test_labels

# Assign review tiers (same thresholds as XGBoost/CatBoost)
results['review_tier'] = pd.cut(
    test_confidence,
    bins=[0, 0.7, 0.9, 1.01],
    labels=['RED', 'YELLOW', 'GREEN']
)

print("CONFIDENCE TIER BREAKDOWN:")
print(f"{'Tier':<10} {'Count':>6} {'Pct':>6} {'Accuracy':>10}")
print("-" * 35)
for tier in ['GREEN', 'YELLOW', 'RED']:
    tier_data = results[results['review_tier'] == tier]
    if len(tier_data) > 0:
        tier_acc = tier_data['correct'].mean()
        pct = len(tier_data) / len(results) * 100
        print(f"{tier:<10} {len(tier_data):>6} {pct:>5.1f}% {tier_acc:>9.1%}")

CONFIDENCE TIER BREAKDOWN:
Tier        Count    Pct   Accuracy
-----------------------------------
GREEN         138  73.8%     99.3%
YELLOW         15   8.0%     66.7%
RED            34  18.2%     79.4%


In [20]:
# Display test predictions for Data Wrangler verification
print(f"\nTest Predictions DataFrame: {len(results)} rows\n")

# Select key columns for verification
output_cols = [
    'date', 'description', 'vendor_name', 'amount',
    'category_true', 'predicted', 'confidence', 'review_tier', 'correct'
]

# Display the DataFrame
temp_data = pd.DataFrame(results[output_cols])
temp_data


Test Predictions DataFrame: 187 rows



,date,description,vendor_name,amount,category_true,predicted,confidence,review_tier,correct
0,2023-04-01,PL NOISE MEDIA LLC / Bomb Playlist (FKA Galua)...,PL NOISE MEDIA LLC / Bomb Playlist (FKA Galua),50.00,Bill,Bill,0.996257,GREEN,True
1,2023-02-24,Medicare - employer tax,,64.79,Journal Entry,Journal Entry,0.995843,GREEN,True
2,2023-01-30,LA Webfile E-filing LA WEB EFILINGEC 844-663-4...,LA Webfile E-filing,1196.60,Expense,Expense,0.593541,RED,True
3,2023-07-25,Upwork Upwork -492888977REF,Upwork,84.00,Expense,Expense,0.999830,GREEN,True
4,2023-05-20,"You Grow Groningen Glass Mansions - ""Nearsighted""",You Grow Groningen,563.00,Expense,Expense,0.952164,GREEN,True
...,...,...,...,...,...,...,...,...,...
182,2023-11-07,Google Domains GOOGLE *Domains Mountain View G...,Google Domains,20.45,Expense,Expense,0.952839,GREEN,True
183,2023-09-26,Upwork Upwork -492888977REF,Upwork,26.25,Expense,Expense,0.999792,GREEN,True
184,2023-11-10,CLOUDFLARE PAYPAL *CLOUDFLARE,CLOUDFLARE,10.10,Expense,Expense,0.897998,YELLOW,True
185,2023-05-29,iPostal1 PAYPAL *IPOSTAL1,iPostal1,20.00,Expense,Expense,0.966195,GREEN,True


## Predict on New CSV

In [21]:
# Enhanced prediction function with 5-layer post-prediction validation
def predict_new_csv(csv_path):
    """Load a new CSV and predict categories using cascade + 5-layer validation system"""

    # Load new data
    new_df = pd.read_csv(csv_path)

    # Find memo/description column
    memo_cols = [c for c in new_df.columns if 'memo' in c.lower() or 'description' in c.lower()]
    memo_col = memo_cols[0] if memo_cols else None

    # Build description (vendor + memo)
    new_df['vendor_name'] = new_df['Name'].fillna('').str.strip()
    memo_text = new_df[memo_col].fillna('').str.strip() if memo_col else ''
    new_df['description'] = (new_df['vendor_name'] + ' ' + memo_text).str.strip()
    new_df['amount'] = new_df['Debit'].fillna(0) + new_df['Credit'].fillna(0)

    # Classify account types (for validation layer 2)
    new_df[['account_type', 'account_code', 'account_name']] = new_df['Account'].apply(
        lambda x: pd.Series(classify_account(x))
    )

    # Apply rule-based classification FIRST (Layer 1)
    new_rules = apply_rule_based_classification(new_df)
    rule_matches = new_rules['has_rule_match'] == 1
    rule_high_conf = new_rules['rule_confidence'] >= 0.85

    print(f"Rule-based pre-processing:")
    print(f"  {rule_matches.sum()} transactions matched by rules ({rule_matches.mean()*100:.1f}%)")
    print(f"  {(rule_matches & rule_high_conf).sum()} with high confidence (≥85%)")

    # TF-IDF features
    new_tfidf = tfidf.transform(new_df['description'])
    new_features = pd.DataFrame(new_tfidf.toarray(), columns=tfidf_cols)
    new_features['amount_log'] = np.log1p(new_df['amount'])

    # Vendor Intelligence
    new_vi = apply_vi(new_df, vi)

    # Transportation features
    new_transport = add_transportation_features(new_df)

    # Rule-based features (numeric only)
    new_rules_numeric = new_rules[['rule_confidence', 'has_rule_match']]

    # Combine and select features for ML model
    new_combined = pd.concat([new_features, new_vi, new_transport, new_rules_numeric], axis=1)
    if best_k < train_combined.shape[1]:
        new_final = selector.transform(new_combined)
    else:
        new_final = new_combined

    # ML predictions for ALL transactions
    ml_predictions = nb_final.predict(new_final)
    ml_probabilities = nb_final.predict_proba(new_final)
    ml_confidence = ml_probabilities.max(axis=1)

    # HYBRID: Use rule predictions if high confidence, otherwise use ML
    final_predictions = []
    final_confidence = []
    prediction_source = []

    for idx in range(len(new_df)):
        if rule_matches.iloc[idx] and rule_high_conf.iloc[idx]:
            # Use rule-based prediction (high confidence)
            final_predictions.append(new_rules.iloc[idx]['rule_prediction'])
            final_confidence.append(new_rules.iloc[idx]['rule_confidence'])
            prediction_source.append('rule')
        else:
            # Use ML prediction
            final_predictions.append(ml_predictions[idx])
            final_confidence.append(ml_confidence[idx])
            prediction_source.append('ml')

    # Add predictions to dataframe
    new_df['Transaction Type'] = final_predictions
    new_df['Confidence Score'] = final_confidence
    new_df['Confidence Tier'] = pd.cut(final_confidence, bins=[0, 0.7, 0.9, 1.01],
                                        labels=['RED', 'YELLOW', 'GREEN'])
    new_df['Prediction Source'] = prediction_source

    # Report breakdown BEFORE validation
    print(f"\nPrediction source breakdown (pre-validation):")
    print(f"  Rule-based: {(new_df['Prediction Source']=='rule').sum()} ({(new_df['Prediction Source']=='rule').mean()*100:.1f}%)")
    print(f"  ML model: {(new_df['Prediction Source']=='ml').sum()} ({(new_df['Prediction Source']=='ml').mean()*100:.1f}%)")

    pre_validation_tiers = new_df['Confidence Tier'].value_counts().to_dict()
    print(f"\nPre-validation tiers: {pre_validation_tiers}")

    # ========================================================================
    # POST-PREDICTION VALIDATION (5 LAYERS)
    # ========================================================================
    print(f"\n{'='*70}")
    print("APPLYING POST-PREDICTION VALIDATION (5 LAYERS)")
    print(f"{'='*70}")

    validator = PostPredictionValidator()
    validated_df = validator.validate_batch(
        new_df,
        prediction_col='Transaction Type',
        confidence_col='Confidence Score',
        amount_col='amount',
        description_col='description',
        vendor_col='vendor_name',
        account_type_col='account_type'
    )

    # Get validation statistics
    val_stats = validator.get_validation_summary()
    print(f"\nValidation Layer Impact:")
    print(f"  Layer 1 - Business Logic Overrides: {val_stats['business_logic_overrides']}")
    print(f"  Layer 2 - Account Alignment Warnings: {val_stats['account_alignment_warnings']}")
    print(f"  Layer 3 - Consistency Flags: {val_stats['consistency_flags']}")
    print(f"  Layer 4 - Confidence Boosts: {val_stats['confidence_boosts']}")
    print(f"  Layer 4 - Confidence Penalties: {val_stats['confidence_penalties']}")
    if 'transportation_overrides' in val_stats:
        print(f"  Layer 5 - Transportation Overrides: {val_stats['transportation_overrides']}")

    # Show tier changes
    post_validation_tiers = validated_df['Validated Tier'].value_counts().to_dict()
    print(f"\nPost-validation tiers: {post_validation_tiers}")

    # Calculate tier changes
    for tier in ['RED', 'YELLOW', 'GREEN']:
        pre_count = pre_validation_tiers.get(tier, 0)
        post_count = post_validation_tiers.get(tier, 0)
        change = post_count - pre_count
        if change != 0:
            sign = '+' if change > 0 else ''
            print(f"  {tier}: {pre_count} → {post_count} ({sign}{change})")

    # Count prediction overrides
    overrides = (validated_df['Transaction Type'] != validated_df['Validated Transaction Type']).sum()
    print(f"\nPrediction overrides: {overrides} ({overrides/len(validated_df)*100:.1f}%)")

    # Show average confidence delta
    avg_delta = validated_df['Confidence Delta'].mean()
    print(f"Average confidence delta: {avg_delta:+.4f}")

    print(f"{'='*70}\n")

    return validated_df

# Example usage: Predict on new CSV and display as DataFrame
new_predictions = predict_new_csv('/Users/gagepiercegaubert/Desktop/career_projects/co-keeper/CSV_data/by_year_edits/General_ledger_2025_edited.csv')

print(f"✓ Generated {len(new_predictions)} predictions")
print(f"Final tier breakdown: {new_predictions['Validated Tier'].value_counts().to_dict()}")

# Display DataFrame for Data Wrangler
new_predictions

Rule-based pre-processing:
  498 transactions matched by rules (20.8%)
  284 with high confidence (≥85%)

Prediction source breakdown (pre-validation):
  Rule-based: 284 (11.9%)
  ML model: 2106 (88.1%)

Pre-validation tiers: {'GREEN': 1001, 'RED': 923, 'YELLOW': 466}

APPLYING POST-PREDICTION VALIDATION (5 LAYERS)

Validation Layer Impact:
  Layer 1 - Business Logic Overrides: 215
  Layer 2 - Account Alignment Warnings: 791
  Layer 3 - Consistency Flags: 0
  Layer 4 - Confidence Boosts: 1031
  Layer 4 - Confidence Penalties: 800
  Layer 5 - Transportation Overrides: 9

Post-validation tiers: {'YELLOW': 832, 'GREEN': 826, 'RED': 732}
  RED: 923 → 732 (-191)
  YELLOW: 466 → 832 (+366)
  GREEN: 1001 → 826 (-175)

Prediction overrides: 19 (0.8%)
Average confidence delta: -0.0172

✓ Generated 2390 predictions
Final tier breakdown: {'YELLOW': 832, 'GREEN': 826, 'RED': 732}


,Date,Transaction Type,Num,Name,Memo/Description,Account,Debit,Credit,Balance,vendor_name,...,account_code,account_name,Confidence Score,Confidence Tier,Prediction Source,Validated Transaction Type,Validated Confidence,Validation Flags,Confidence Delta,Validated Tier
0,01/31/2025,Expense,NaN,NaN,NaN,46497910 Vanguard Brokerage,138.42,NaN,41714.74,,...,46497910,Vanguard Brokerage,0.656794,RED,ml,Expense,0.656794,,0.00,RED
1,02/05/2025,Expense,NaN,NaN,VANGUARD BUY INDIVIDUAL BUY INVESTMENT VANGUAR...,46497910 Vanguard Brokerage,5500.00,NaN,47214.74,,...,46497910,Vanguard Brokerage,0.537168,RED,ml,Expense,0.637168,transport_airline_match,0.10,RED
2,02/10/2025,Expense,NaN,NaN,VANGUARD SELL INDIVIDUAL SELL INVESTMENT,46497910 Vanguard Brokerage,NaN,2628.00,44586.74,,...,46497910,Vanguard Brokerage,0.565450,RED,ml,Expense,0.665450,transport_airline_match,0.10,RED
3,02/24/2025,Expense,NaN,NaN,VANGUARD BUY INDIVIDUAL BUY INVESTMENT VANGUAR...,46497910 Vanguard Brokerage,650.00,NaN,45236.74,,...,46497910,Vanguard Brokerage,0.613640,RED,ml,Expense,0.713640,transport_airline_match,0.10,YELLOW
4,03/06/2025,Invoice,NaN,NaN,Band of Silver Invoice Transfer,46497910 Vanguard Brokerage,77813.00,NaN,123049.74,,...,46497910,Vanguard Brokerage,0.766997,YELLOW,ml,Invoice,0.816997,large_amount_reasonable,0.05,YELLOW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2385,12/05/2025,Expense,NaN,NaN,TST*STICKY RICE - ECHO - 1785,67000 R&D,67.34,NaN,23830.39,,...,67000,R&D,0.672555,RED,ml,Expense,0.752555,account_alignment_match,0.08,YELLOW
2386,12/24/2025,Expense,NaN,NaN,PP*METAPLATFOR,67000 R&D,362.29,NaN,24192.68,,...,67000,R&D,0.662947,RED,ml,Expense,0.842947,"transport_public_transit_match,account_alignme...",0.18,YELLOW
2387,12/26/2025,Invoice,NaN,NaN,"Settle ""Spotify"" Test Placement",67000 R&D,25.00,NaN,24217.68,,...,67000,R&D,0.583023,RED,ml,Invoice,0.433023,account_mismatch_EXPENSE_Invoice,-0.15,RED
2388,01/31/2025,Expense,NaN,NaN,Vanguard Interest,71100 Dividend Income,NaN,138.42,2984.84,,...,71100,Dividend Income,0.656794,RED,ml,Expense,0.736794,account_alignment_match,0.08,YELLOW


## Step 8: Comparison with XGBoost and CatBoost

In [22]:
# Create comparison table
comparison_data = {
    'Model': ['XGBoost', 'CatBoost', 'Naive Bayes (Optimized)'],
    'Test Accuracy': ['91.1%', '88.8%', f"{test_acc_final:.1%}"],
    'Val Accuracy': ['87.9%', '87.4%', f"{val_acc_final:.1%}"],
    'Training Speed': ['Medium', 'Slow', 'Very Fast'],
    'Inference Speed': ['Fast', 'Fast', 'Very Fast'],
    'Features': [500, 500, best_k]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Performance gap analysis
xgb_acc = 0.911
gap = (xgb_acc - test_acc_final) * 100
print(f"\nAccuracy gap from XGBoost: {gap:.1f} percentage points")

if test_acc_final >= 0.85:
    print("✓ TARGET ACHIEVED: 85%+ accuracy reached!")
elif test_acc_final >= 0.80:
    print("⚠ Close to target: 80-85% range")
else:
    print("❌ Target missed: <80% accuracy")


MODEL COMPARISON
                  Model Test Accuracy Val Accuracy Training Speed Inference Speed  Features
                XGBoost         91.1%        87.9%         Medium            Fast       500
               CatBoost         88.8%        87.4%           Slow            Fast       500
Naive Bayes (Optimized)         93.0%        94.1%      Very Fast       Very Fast       100

Accuracy gap from XGBoost: -1.9 percentage points
✓ TARGET ACHIEVED: 85%+ accuracy reached!


## Step 9: Sample Predictions Analysis

In [23]:
# Show sample predictions from each tier
print("\nSAMPLE PREDICTIONS:")
print("="*90)

for tier in ['GREEN', 'YELLOW', 'RED']:
    tier_data = results[results['review_tier'] == tier].head(3)
    if len(tier_data) > 0:
        print(f"\n{tier} Tier Examples:")
        for _, row in tier_data.iterrows():
            status = "✓" if row['correct'] else "✗"
            print(f"  {status} {row['description'][:50]:50s}")
            print(f"     Predicted: {row['predicted'][:40]:40s} ({row['confidence']:.1%})")
            print(f"     Actual:    {row['category_true'][:40]:40s}")


SAMPLE PREDICTIONS:

GREEN Tier Examples:
  ✓ PL NOISE MEDIA LLC / Bomb Playlist (FKA Galua) Soc
     Predicted: Bill                                     (99.6%)
     Actual:    Bill                                    
  ✓ Medicare - employer tax                           
     Predicted: Journal Entry                            (99.6%)
     Actual:    Journal Entry                           
  ✓ Upwork Upwork -492888977REF                       
     Predicted: Expense                                  (100.0%)
     Actual:    Expense                                 

YELLOW Tier Examples:
  ✓ Phil Makes:Phil Makes - "Winston" (Spotify) Phil M
     Predicted: Invoice                                  (76.1%)
     Actual:    Invoice                                 
  ✗ Next Insurance NEXT Insur (AP I NEXT Insur ID NBR:
     Predicted: Deposit                                  (85.2%)
     Actual:    Expense                                 
  ✓ Jono Robertson:Social Creatures - "Half the 

## Summary & Recommendations

### Key Findings:
1. **Baseline Performance**: MultinomialNB and ComplementNB baselines
2. **Alpha Optimization**: Best smoothing parameter found
3. **Feature Selection**: Optimal K features identified
4. **Final Accuracy**: Comparison against gradient boosting models

### Next Steps:
- If accuracy <85%: Try ensemble methods (combine with RandomForest)
- If accuracy 85-88%: Consider hybrid approach (NB for text, GBM for complex patterns)
- If accuracy >88%: Naive Bayes is production-ready!

### Trade-offs:
- **Speed**: Naive Bayes is 10-100x faster than gradient boosting
- **Accuracy**: Expected 5-10% lower than XGBoost
- **Simplicity**: Much easier to interpret and debug
- **Use Case**: Great for real-time predictions, may need review queue for low confidence